# Supplementary Plotting Rewrite

This notebook regenerates Supplementary Information source figures using the style of `final_report_plotting_examples.ipynb`.

Fast plotting sections are enabled by default. Long retraining sections are controlled by explicit flags so the notebook can be opened and run safely; when enabled, all training outputs are written under `results/supplementary_rewrite/` and all source figures under `si_figure_sources_rewrite/`.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import math
import os
import re
import shutil
import sys
import time
from typing import Callable

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import NullFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.interpolate import griddata
from IPython.display import display

try:
    import umap.umap_ as umap
    HAS_UMAP = True
except Exception as exc:
    HAS_UMAP = False
    UMAP_IMPORT_ERROR = exc

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIGURE_ROOT = PROJECT_ROOT / 'si_figure_sources_rewrite'
RESULT_ROOT = PROJECT_ROOT / 'results' / 'supplementary_rewrite'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# Fast sections.
RUN_MEASUREMENT_EXPORT = True
RUN_SIMULATION_DISPLAY_EXPORT = True
RUN_EXISTING_ML_EXPORT = True
RUN_FINAL_OUTLIER_EXPORT = True
RUN_UMAP_EXPORT = True
EXPORT_SI_COMPOSITES = False
WRITEUP_ROOT = os.environ.get('XPCS_WRITEUP_ROOT')
WRITEUP_ROOT = Path(WRITEUP_ROOT).expanduser().resolve() if WRITEUP_ROOT else None

# Long sections. Enable only when regenerating the retraining controls.
RUN_AUGMENTATION_RETRAINING = False
RUN_WEIGHT_SWEEP = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42
print('project:', PROJECT_ROOT)
print('figures:', FIGURE_ROOT)
print('results:', RESULT_ROOT)
if WRITEUP_ROOT is not None:
    print('writeup:', WRITEUP_ROOT)
print('device:', DEVICE)
if not HAS_UMAP:
    print('UMAP import failed:', repr(UMAP_IMPORT_ERROR))


## Shared Report Style And Data Helpers


In [ ]:
def set_report_style(base_font_size: int = 28) -> None:
    plt.rcParams['font.family'] = 'arial'
    plt.rcParams['font.size'] = base_font_size
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'arial'
    plt.rcParams['mathtext.it'] = 'arial:italic'
    plt.rcParams['mathtext.bf'] = 'arial:bold'
    plt.rcParams['mathtext.cal'] = 'cmsy10'
    plt.rcParams['axes.linewidth'] = 1.2
    plt.rcParams['xtick.major.width'] = 1.2
    plt.rcParams['ytick.major.width'] = 1.2

set_report_style()

SLICE_TIMES_S = [0, 500, 1000, 1500, 2000]
SLICE_COLORS = ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e']
TOTAL_TIME_S = 2500.0
TIME_TICKS_S = [0, 1250, 2500]
TIME_TICK_LABELS = ['0', '1250', '2500']


def as_float_array(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().squeeze().numpy()
    return np.asarray(x, dtype=np.float32)


def coarsen_square(x, target_size: int | None = 256):
    x = as_float_array(x)
    if target_size is None or x.shape[0] == target_size:
        return x
    row_idx = np.linspace(0, x.shape[0] - 1, target_size).round().astype(int)
    col_idx = np.linspace(0, x.shape[1] - 1, target_size).round().astype(int)
    return np.asarray(x[np.ix_(row_idx, col_idx)], dtype=np.float32)


def normalize_g2_report(g2, min_val=1.0, max_val=1.2, diagonal_value=1.2, ignore_diagonal=True):
    tensor = torch.as_tensor(as_float_array(g2), dtype=torch.float32).clone()
    ref = tensor
    if ignore_diagonal and tensor.ndim == 2 and tensor.shape[0] == tensor.shape[1]:
        mask = ~torch.eye(tensor.shape[0], dtype=torch.bool, device=tensor.device)
        ref = tensor[mask]
    g2_min, g2_max = ref.min(), ref.max()
    out = (tensor - g2_min) / (g2_max - g2_min + 1e-8) * (max_val - min_val) + min_val
    if diagonal_value is not None and out.ndim == 2 and out.shape[0] == out.shape[1]:
        out.fill_diagonal_(diagonal_value)
    return as_float_array(out)


def normalize_simulation_panel(g2):
    tensor = torch.as_tensor(as_float_array(g2), dtype=torch.float32)
    g2_min, g2_max = tensor.min(), tensor.max()
    return as_float_array((tensor - g2_min) / (g2_max - g2_min + 1e-8) * 0.2 + 1.0)


def diagonal_mean_matrix(x):
    x = as_float_array(x)
    n = x.shape[0]
    out = np.empty_like(x)
    for offset in range(-n + 1, n):
        mean_value = float(np.diagonal(x, offset=offset).mean())
        if offset >= 0:
            idx = np.arange(n - offset)
            out[idx, idx + offset] = mean_value
        else:
            idx = np.arange(n + offset)
            out[idx - offset, idx] = mean_value
    return out


def nonequilibrium_measure_np(g2):
    g2 = as_float_array(g2)
    residual = g2 - diagonal_mean_matrix(g2)
    denominator = g2 - g2.mean()
    return float(np.linalg.norm(residual) / (np.linalg.norm(denominator) + 1e-12))


def time_to_index(time_s: float, n: int, total_time_s: float = TOTAL_TIME_S) -> int:
    return int(np.clip(round(time_s / total_time_s * (n - 1)), 0, n - 1))


def time_axis_s(n: int, total_time_s: float = TOTAL_TIME_S):
    return np.linspace(0.0, total_time_s, n)


def parse_temperature_c(path: Path) -> float:
    match = re.search(r'_T(-?\d+(?:\.\d+)?)C$', path.stem)
    if match is None:
        raise ValueError(f'Could not parse temperature from {path.name}')
    return float(match.group(1))


def experiment_group_name(path: Path) -> str:
    match = re.match(r'(.+)_T-?\d+(?:\.\d+)?C$', path.stem)
    if match is None:
        raise ValueError(f'Could not parse group from {path.name}')
    return match.group(1)


def load_raw_g2_map(path: Path, channel: int = 0, t_max_s: int = 2500):
    path = Path(path)
    if path.suffix.lower() == '.npz':
        with np.load(path) as data:
            g2_all = data['g12']
            channel = min(channel, g2_all.shape[-1] - 1) if g2_all.ndim == 3 else 0
            stop = min(int(t_max_s) + 1, g2_all.shape[0], g2_all.shape[1])
            g2 = np.asarray(g2_all[:stop, :stop, channel] if g2_all.ndim == 3 else g2_all[:stop, :stop], dtype=np.float32)
    elif path.suffix.lower() == '.npy':
        raw = np.load(path, mmap_mode='r')
        channel = min(channel, raw.shape[-1] - 1) if raw.ndim == 3 else 0
        stop = min(int(t_max_s) + 1, raw.shape[0], raw.shape[1])
        g2 = np.asarray(raw[:stop, :stop, channel] if raw.ndim == 3 else raw[:stop, :stop], dtype=np.float32)
    else:
        raise ValueError(f'Unsupported raw experiment file: {path}')
    return g2


def load_experiment_panel(path: Path, channel: int = 0, t_max_s: int = 2500, coarse_size: int = 256):
    raw = load_raw_g2_map(path, channel=channel, t_max_s=t_max_s)
    panel = coarsen_square(raw, target_size=coarse_size)
    panel = normalize_g2_report(panel, min_val=1.0, max_val=1.2, diagonal_value=1.2, ignore_diagonal=True)
    return panel


def load_simulation_panel(path: Path, t_max_s: int = 2500):
    g2 = torch.load(path, map_location='cpu', weights_only=True).squeeze(0)
    return normalize_simulation_panel(g2)


def plot_g2_panel_row(panels, output_path: Path, *, title_key='label', cmap='viridis', vmin=1.0, vmax=1.2, figsize=None, aspect='equal'):
    if figsize is None:
        figsize = (7.5 * len(panels) + 2, 6)
    fig, axes = plt.subplots(1, len(panels), figsize=figsize, sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    image = None
    for ax, panel in zip(axes, panels):
        image = ax.imshow(panel['g2'], origin='lower', vmin=vmin, vmax=vmax, cmap=cmap, interpolation='nearest', rasterized=True, extent=(0, TOTAL_TIME_S, 0, TOTAL_TIME_S), aspect=aspect)
        if title_key is not None and panel.get(title_key):
            ax.set_title(panel[title_key], pad=10)
        ax.set_xlabel(r'Time $t_2$ (s)')
        ax.set_xticks(TIME_TICKS_S, TIME_TICK_LABELS)
        ax.set_yticks(TIME_TICKS_S, TIME_TICK_LABELS)
    axes[0].set_ylabel(r'Time $t_1$ (s)')
    colorbar = fig.colorbar(image, ax=axes, location='right', pad=0.035)
    colorbar.set_label(r'$g_2(t_1,t_2)$')
    colorbar.set_ticks([1.0, 1.1, 1.2])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches='tight')
    display(fig)
    plt.close(fig)


def plot_single_g2_panel(panel, output_path: Path, *, cmap='viridis', vmin=1.0, vmax=1.2):
    fig, ax = plt.subplots(figsize=(5.85, 6))
    ax.imshow(panel['g2'], origin='lower', vmin=vmin, vmax=vmax, cmap=cmap, interpolation='nearest', rasterized=True, extent=(0, TOTAL_TIME_S, 0, TOTAL_TIME_S), aspect='auto')
    ax.set_xlabel(r'Time $t_2$ (s)')
    ax.set_ylabel(r'Time $t_1$ (s)')
    ax.set_xticks(TIME_TICKS_S, TIME_TICK_LABELS)
    ax.set_yticks(TIME_TICKS_S, TIME_TICK_LABELS)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches='tight')
    display(fig)
    plt.close(fig)


def plot_autocorr_panel_row(panels, output_path: Path, *, title_key='label', figsize=None):
    if figsize is None:
        figsize = (7.0 * len(panels) + 3, 6)
    fig, axes = plt.subplots(1, len(panels), figsize=figsize, sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    for ax, panel in zip(axes, panels):
        n = panel['g2'].shape[0]
        for t1_s, color in zip(SLICE_TIMES_S, SLICE_COLORS):
            i = time_to_index(t1_s, n)
            tau_s = np.arange(n - i) / max(n - 1, 1) * TOTAL_TIME_S
            curve = panel['g2'][i, i:]
            ax.plot(tau_s, curve, lw=2, color=color, label=rf'$t_1 =${t1_s:g} s')
        if title_key is not None and panel.get(title_key):
            ax.set_title(panel[title_key], pad=10)
        ax.set_xlabel(r'Lag time $\tau$ (s)')
        ax.set_xlim(0, TOTAL_TIME_S)
        ax.set_xticks(TIME_TICKS_S, TIME_TICK_LABELS)
        ax.set_ylim(0.98, 1.22)
    axes[0].set_ylabel(r'$f(\tau;t_1)=g_2(t_1,t_1+\tau)$')
    axes[-1].legend(frameon=True, loc='upper left', bbox_to_anchor=(1.02, 1.0))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches='tight')
    display(fig)
    plt.close(fig)


## 1. Measurement Candidates

All top-level `.npy`/`.npz` experimental groups in `exp_data/` are discovered automatically. The main-text group is excluded. If a group has more than three temperatures, the low / middle / high representative temperatures are used.


In [ ]:
MEASUREMENT_OUTPUT_DIR = FIGURE_ROOT / 'FigS_measurement_candidates'
MEASUREMENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCLUDE_EXPERIMENT_GROUPS = {'2NSN10_16_2019_dose0'}
MIN_TEMPERATURES_PER_GROUP = 3


def discover_experiment_groups(exp_data_dir=PROJECT_ROOT / 'exp_data'):
    rows = []
    for path in sorted(Path(exp_data_dir).iterdir()):
        if path.suffix.lower() not in {'.npy', '.npz'}:
            continue
        try:
            group = experiment_group_name(path)
            temp_c = parse_temperature_c(path)
        except ValueError:
            continue
        rows.append({'group': group, 'temperature_c': temp_c, 'temperature_k': temp_c + 273.15, 'path': path})
    table = pd.DataFrame(rows).sort_values(['group', 'temperature_c']).reset_index(drop=True)
    return table


def choose_representative_temperatures(group_df, n=3):
    group_df = group_df.sort_values('temperature_c').reset_index(drop=True)
    if len(group_df) <= n:
        return group_df
    indices = np.linspace(0, len(group_df) - 1, n).round().astype(int)
    return group_df.iloc[sorted(set(indices))].reset_index(drop=True)

experiment_file_table = discover_experiment_groups()
measurement_groups = []
for group, group_df in experiment_file_table.groupby('group', sort=True):
    if group in EXCLUDE_EXPERIMENT_GROUPS or len(group_df) < MIN_TEMPERATURES_PER_GROUP:
        continue
    selected = choose_representative_temperatures(group_df, n=3)
    measurement_groups.append(selected.assign(selected_for_plot=True))
measurement_selection = pd.concat(measurement_groups, ignore_index=True) if measurement_groups else pd.DataFrame()
measurement_selection.to_csv(MEASUREMENT_OUTPUT_DIR / 'measurement_group_temperature_selection.csv', index=False)
measurement_selection[['group', 'temperature_c', 'temperature_k', 'path']]


In [ ]:
if RUN_MEASUREMENT_EXPORT:
    measurement_summary = []
    for group, selected in measurement_selection.groupby('group', sort=True):
        panels = []
        for _, row in selected.sort_values('temperature_c').iterrows():
            g2 = load_experiment_panel(Path(row['path']), channel=0, t_max_s=2500, coarse_size=256)
            panels.append({
                'label': f"{row['temperature_k']:.0f} K",
                'g2': g2,
                'path': Path(row['path']),
                'temperature_c': float(row['temperature_c']),
                'S_noneq': nonequilibrium_measure_np(g2),
            })
        safe_group = re.sub(r'[^A-Za-z0-9_.-]+', '_', group)
        plot_g2_panel_row(panels, MEASUREMENT_OUTPUT_DIR / f'{safe_group}_g2.pdf')
        plot_autocorr_panel_row(panels, MEASUREMENT_OUTPUT_DIR / f'{safe_group}_autocorr.pdf')
        for panel in panels:
            measurement_summary.append({
                'group': group,
                'temperature_c': panel['temperature_c'],
                'temperature_k_label': panel['label'],
                'path': str(panel['path']),
                'S_noneq_plot_normalized': panel['S_noneq'],
            })
    measurement_summary = pd.DataFrame(measurement_summary)
    measurement_summary.to_csv(MEASUREMENT_OUTPUT_DIR / 'measurement_summary.csv', index=False)
    display(measurement_summary)
else:
    print('Measurement export disabled.')


## 2. Simulation Display: Clean Phase Map And Fixed-`lambda_GB` Selections

The phase diagram is saved as PNG at 144 dpi. Three target `lambda_GB` levels are chosen manually, and four points are selected as the closest available simulated cases to each target level from the same fixed raw non-equilibrium-measure regions used in `final_report_plotting_examples.ipynb`. Because the simulation manifest is continuously sampled rather than gridded in `GB_conc`, the selected rows are near-horizontal rather than exact repeated `GB_conc` values.


In [ ]:
SIM_OUTPUT_DIR = FIGURE_ROOT / 'FigS_simulation_lambda_D'
SIM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SIM_MANIFEST = PROJECT_ROOT / 'dataset' / 'simulation_2' / 'manifest_with_non_equ.csv'
SIM_LAMBDA_TARGETS = [0.12, 0.24, 0.36]
SIM_LAMBDA_INITIAL_BAND = 0.025
SIM_LAMBDA_MAX_BAND = 0.09
SIM_CLOSEST_LAMBDA_CANDIDATES = 500
SIM_LAMBDA_SELECTION_SCALE = 0.025
SIM_D_MIN_LOG_GAP = 0.28
SIM_D_ANCHOR_WEIGHT = 1.0
SIM_NONEQ_CENTER_WEIGHT = 0.03
SIM_NONEQ_REGIONS = [
    (0.0, 0.2),
    (0.2, 0.4),
    (0.4, 0.7),
    (0.7, 1.0),
]
SIM_SELECTION_RNG_SEED = 20050628
SIM_POINT_COLORS = ['#009E73', '#0072B2', '#D55E00']
SIM_MARKER_SIZE = 420
SIM_MARKER_EDGE = 'white'

simulation_df = pd.read_csv(SIM_MANIFEST).copy()
if 'nonequilibrium_measure_raw' not in simulation_df.columns and 'nonequilibrium_measure' in simulation_df.columns:
    simulation_df['nonequilibrium_measure_raw'] = simulation_df['nonequilibrium_measure']
simulation_df = simulation_df.loc[(simulation_df['D'] > 0) & simulation_df['GB_conc'].between(0, 0.5)].reset_index(drop=True)


def interpolate_phase_grid(df, x='D', y='GB_conc', value='nonequilibrium_measure_raw', xscale='log', yscale='linear', x_range=(3e-24, 3e-21), y_range=(0.0, 0.5), resolution=400):
    x_values = df[x].to_numpy(dtype=float)
    y_values = df[y].to_numpy(dtype=float)
    z_values = df[value].to_numpy(dtype=float)
    x_transformed = np.log10(x_values) if xscale == 'log' else x_values
    y_transformed = np.log10(y_values) if yscale == 'log' else y_values
    xt_lo = np.log10(x_range[0]) if xscale == 'log' else x_range[0]
    xt_hi = np.log10(x_range[1]) if xscale == 'log' else x_range[1]
    yt_lo = np.log10(y_range[0]) if yscale == 'log' else y_range[0]
    yt_hi = np.log10(y_range[1]) if yscale == 'log' else y_range[1]
    grid_x_t, grid_y_t = np.meshgrid(np.linspace(xt_lo, xt_hi, resolution), np.linspace(yt_lo, yt_hi, resolution))
    grid_z = griddata(np.column_stack([x_transformed, y_transformed]), z_values, (grid_x_t, grid_y_t), method='linear', rescale=True)
    nearest = griddata(np.column_stack([x_transformed, y_transformed]), z_values, (grid_x_t, grid_y_t), method='nearest', rescale=True)
    grid_z = np.where(np.isfinite(grid_z), grid_z, nearest)
    grid_z = np.clip(grid_z, 0.0, 1.0)
    grid_x = np.power(10.0, grid_x_t) if xscale == 'log' else grid_x_t
    grid_y = np.power(10.0, grid_y_t) if yscale == 'log' else grid_y_t
    return grid_x, grid_y, grid_z


def plot_D_GB_phase(output_path, selected_points=None, clean=False, x_range=(3e-24, 3e-21), y_range=(0.0, 0.5), figsize=(11.2, 9.8), dpi=144):
    grid_x, grid_y, grid_z = interpolate_phase_grid(simulation_df, x_range=x_range, y_range=y_range)
    fig, ax = plt.subplots(figsize=figsize)
    contour = ax.contourf(grid_x, grid_y, grid_z, levels=np.linspace(0, 1, 101), cmap='plasma', norm=Normalize(vmin=0.0, vmax=1.0))
    for collection in getattr(contour, 'collections', []):
        collection.set_rasterized(True)
    if selected_points is not None and len(selected_points):
        for lambda_index, group in selected_points.groupby('lambda_index', sort=True):
            color = SIM_POINT_COLORS[int(lambda_index) % len(SIM_POINT_COLORS)]
            ax.scatter(group['D'], group['GB_conc'], s=SIM_MARKER_SIZE, marker='o', facecolors=color, edgecolors=SIM_MARKER_EDGE, linewidths=1.4, zorder=5)
    ax.set_xscale('log')
    ax.set_xlim(*x_range)
    ax.set_ylim(*y_range)
    ax.set_xlabel(r'Diffusivity $D$ (cm$^2\cdot$s$^{-1}$)')
    ax.set_ylabel(r'Effective GB concentration $\lambda_{\mathrm{GB}}$')
    ax.set_xticks([1e-23, 1e-22, 1e-21], [r'$10^{-23}$', r'$10^{-22}$', r'$10^{-21}$'])
    ax.set_yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5], ['0', '0.1', '0.2', '0.3', '0.4', '0.5'])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size=0.32, pad=0.35)
    cbar = fig.colorbar(contour, cax=cax)
    cbar.set_label('Non-equilibrium measure')
    cbar.set_ticks([0.0, 0.5, 1.0])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches='tight', dpi=dpi)
    display(fig)
    plt.close(fig)


def simulation_region_candidates(df, target, region_index, region_min, region_max, noneq_regions=SIM_NONEQ_REGIONS):
    values = df['nonequilibrium_measure_raw']
    if region_index == len(noneq_regions):
        in_region = values.between(region_min, region_max, inclusive='both')
    else:
        in_region = (values >= region_min) & (values < region_max)
    candidates = df.loc[in_region].copy().reset_index().rename(columns={'index': 'manifest_row'})
    if candidates.empty:
        raise ValueError(f'No simulation rows found with raw non-equilibrium measure in [{region_min}, {region_max}]')
    candidates['lambda_delta'] = (candidates['GB_conc'] - target).abs()
    candidates['D_log10'] = np.log10(candidates['D'].astype(float))
    center = 0.5 * (region_min + region_max)
    width = max(region_max - region_min, 1e-12)
    candidates['noneq_center_cost'] = ((candidates['nonequilibrium_measure_raw'] - center) / (0.5 * width)) ** 2

    lambda_band = SIM_LAMBDA_INITIAL_BAND
    subset = candidates.loc[candidates['lambda_delta'] <= lambda_band].copy()
    while len(subset) < min(30, len(candidates)) and lambda_band < SIM_LAMBDA_MAX_BAND:
        lambda_band = min(SIM_LAMBDA_MAX_BAND, lambda_band * 1.5)
        subset = candidates.loc[candidates['lambda_delta'] <= lambda_band].copy()
    if subset.empty:
        subset = candidates.nsmallest(SIM_CLOSEST_LAMBDA_CANDIDATES, 'lambda_delta').copy()
        lambda_band = float(subset['lambda_delta'].max())
    elif len(subset) > SIM_CLOSEST_LAMBDA_CANDIDATES:
        nearest_count = SIM_CLOSEST_LAMBDA_CANDIDATES // 2
        spread_count = SIM_CLOSEST_LAMBDA_CANDIDATES - nearest_count
        nearest = subset.nsmallest(nearest_count, 'lambda_delta')
        spread_indices = np.linspace(0, len(subset) - 1, spread_count).round().astype(int)
        spread = subset.sort_values('D_log10').iloc[spread_indices]
        subset = pd.concat([nearest, spread]).drop_duplicates('manifest_row')
    subset['adaptive_lambda_band'] = lambda_band
    return subset.reset_index(drop=True)


def choose_lambda_D_points(df, lambda_targets, noneq_regions=SIM_NONEQ_REGIONS):
    rows = []
    for lambda_index, target in enumerate(lambda_targets):
        region_tables = []
        for region_index, (region_min, region_max) in enumerate(noneq_regions, start=1):
            region_tables.append(simulation_region_candidates(df, target, region_index, region_min, region_max, noneq_regions=noneq_regions))

        # Search from high to low non-equilibrium response, which should move from low to high D.
        search_tables = list(reversed(region_tables))
        low_anchor = np.quantile(search_tables[0]['D_log10'], 0.10)
        high_anchor = np.quantile(search_tables[-1]['D_log10'], 0.90)
        if high_anchor <= low_anchor:
            low_anchor = min(table['D_log10'].min() for table in search_tables)
            high_anchor = max(table['D_log10'].max() for table in search_tables)
        D_log10_anchors = np.linspace(low_anchor, high_anchor, len(noneq_regions))

        chosen_high_to_low = []
        previous_logD = -np.inf
        for search_index, table in enumerate(search_tables):
            eligible = table.loc[table['D_log10'] >= previous_logD + SIM_D_MIN_LOG_GAP].copy()
            gap_relaxed = False
            if eligible.empty:
                eligible = table.loc[table['D_log10'] > previous_logD].copy()
                gap_relaxed = True
            if eligible.empty:
                eligible = table.copy()
                gap_relaxed = True

            anchor = D_log10_anchors[search_index]
            eligible['D_anchor_log10'] = anchor
            eligible['selection_cost'] = (
                (eligible['lambda_delta'] / SIM_LAMBDA_SELECTION_SCALE) ** 2
                + SIM_D_ANCHOR_WEIGHT * (eligible['D_log10'] - anchor) ** 2
                + SIM_NONEQ_CENTER_WEIGHT * eligible['noneq_center_cost']
            )
            row = eligible.nsmallest(1, 'selection_cost').iloc[0].copy()
            row['D_gap_constraint_relaxed'] = gap_relaxed
            chosen_high_to_low.append(row)
            previous_logD = float(row['D_log10'])

        best_rows = list(reversed(chosen_high_to_low))
        lambda_selection_score = float(sum(row['selection_cost'] for row in best_rows))
        for region_index, row in enumerate(best_rows, start=1):
            region_min, region_max = noneq_regions[region_index - 1]
            row['lambda_index'] = lambda_index
            row['lambda_target'] = target
            row['lambda_selection_score'] = lambda_selection_score
            row['noneq_region'] = region_index
            row['region_min'] = region_min
            row['region_max'] = region_max
            row['region_label'] = f"R{region_index}: [{region_min:g}, {region_max:g}{']' if region_index == len(noneq_regions) else ')'}"
            row['D_quartile'] = region_index
            rows.append(row)
    selected = pd.DataFrame(rows).sort_values(['lambda_index', 'D']).reset_index(drop=True)
    selected['selection_order'] = selected.groupby('lambda_index').cumcount() + 1
    selected['D_gap_from_previous_log10'] = selected.groupby('lambda_index')['D_log10'].diff()
    return selected

selected_sim_lambda_D = choose_lambda_D_points(simulation_df, SIM_LAMBDA_TARGETS)
selected_sim_lambda_D.to_csv(SIM_OUTPUT_DIR / 'simulation_lambda_D_quartile_selection.csv', index=False)
selected_sim_lambda_D[['lambda_index', 'lambda_target', 'selection_order', 'noneq_region', 'region_label', 'D', 'GB_conc', 'lambda_delta', 'gamma', 'T', 'nonequilibrium_measure_raw', 'path']]


In [ ]:
if RUN_SIMULATION_DISPLAY_EXPORT:
    for obsolete in ['simulation_D_GB_phase_clean.png', 'simulation_D_GB_phase_selected_points.png']:
        (SIM_OUTPUT_DIR / obsolete).unlink(missing_ok=True)
    for lambda_index, lambda_df in selected_sim_lambda_D.groupby('lambda_index', sort=True):
        noneq_columns = [col for col in ['lambda_index', 'lambda_target', 'selection_order', 'noneq_region', 'region_label', 'D', 'GB_conc', 'lambda_delta', 'D_log10', 'D_gap_from_previous_log10', 'D_anchor_log10', 'selection_cost', 'lambda_selection_score', 'gamma', 'T', 'nonequilibrium_measure_raw', 'nonequilibrium_measure', 'path'] if col in lambda_df.columns]
        print(f"simulation lambda_GB target {float(lambda_df['lambda_target'].iloc[0]):.4f} / selected cases")
        display(lambda_df.sort_values('D')[noneq_columns])
        panels = []
        for _, row in lambda_df.sort_values('D').iterrows():
            g2 = load_simulation_panel(PROJECT_ROOT / row['path'])
            panels.append({
                'label': '',
                'g2': g2,
                'D': float(row['D']),
                'GB_conc': float(row['GB_conc']),
                'S_noneq': float(row['nonequilibrium_measure_raw']),
            })
        stem = f"lambda{lambda_index + 1}_target_{float(lambda_df['lambda_target'].iloc[0]):.2f}".replace('.', 'p')
        plot_D_GB_phase(SIM_OUTPUT_DIR / f'{stem}_phase_D_GB.png', selected_points=lambda_df)
        plot_g2_panel_row(panels, SIM_OUTPUT_DIR / f'{stem}_g2.pdf', title_key=None, figsize=(30, 6))
        plot_autocorr_panel_row(panels, SIM_OUTPUT_DIR / f'{stem}_autocorr.pdf', title_key=None, figsize=(26, 6))
else:
    print('Simulation display export disabled.')


## Shared ML Plotting Helpers


In [ ]:
from torch.utils.data import DataLoader, Subset
from run_all import PreparedExperimentDataset, filter_files, iter_raw_experiment_files, prepare_experiment_shots, predict_samples, write_results_csvs
from inference import load_overlay_points, add_overlay_metadata

ML_OUTPUT_DIR = FIGURE_ROOT / 'FigS_ML_controls_and_ablation'
ML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SIM_ROOT = PROJECT_ROOT / 'dataset' / 'simulation'
EXP_RAW_DIR = PROJECT_ROOT / 'exp_data'
PHASE_BACKGROUND_MANIFEST = PROJECT_ROOT / 'dataset' / 'simulation_2' / 'manifest.csv'
phase_background_df = pd.read_csv(PHASE_BACKGROUND_MANIFEST)
if 'nonequilibrium_measure_raw' not in phase_background_df.columns and 'nonequilibrium_measure' in phase_background_df.columns:
    phase_background_df['nonequilibrium_measure_raw'] = phase_background_df['nonequilibrium_measure']


def interpolate_phase_grid(df, x='D', y='GB_conc', value='nonequilibrium_measure_raw', xscale='log', yscale='linear', x_range=(3e-24, 3e-21), y_range=(0.0, 0.5), resolution=400):
    x_values = df[x].to_numpy(dtype=float)
    y_values = df[y].to_numpy(dtype=float)
    z_values = df[value].to_numpy(dtype=float)
    x_transformed = np.log10(x_values) if xscale == 'log' else x_values
    y_transformed = np.log10(y_values) if yscale == 'log' else y_values
    xt_lo = np.log10(x_range[0]) if xscale == 'log' else x_range[0]
    xt_hi = np.log10(x_range[1]) if xscale == 'log' else x_range[1]
    yt_lo = np.log10(y_range[0]) if yscale == 'log' else y_range[0]
    yt_hi = np.log10(y_range[1]) if yscale == 'log' else y_range[1]
    grid_x_t, grid_y_t = np.meshgrid(np.linspace(xt_lo, xt_hi, resolution), np.linspace(yt_lo, yt_hi, resolution))
    source = np.column_stack([x_transformed, y_transformed])
    grid_z = griddata(source, z_values, (grid_x_t, grid_y_t), method='linear', rescale=True)
    nearest = griddata(source, z_values, (grid_x_t, grid_y_t), method='nearest', rescale=True)
    grid_z = np.where(np.isfinite(grid_z), grid_z, nearest)
    grid_z = np.clip(grid_z, 0.0, 1.0)
    grid_x = np.power(10.0, grid_x_t) if xscale == 'log' else grid_x_t
    grid_y = np.power(10.0, grid_y_t) if yscale == 'log' else grid_y_t
    return grid_x, grid_y, grid_z


def r2_score_np(true, pred):
    true = np.asarray(true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    finite = np.isfinite(true) & np.isfinite(pred)
    true = true[finite]
    pred = pred[finite]
    if true.size < 2:
        return np.nan
    ss_res = np.sum((pred - true) ** 2)
    ss_tot = np.sum((true - true.mean()) ** 2)
    return np.nan if ss_tot <= 0 else float(1.0 - ss_res / ss_tot)


def slugify(label):
    return re.sub(r'[^A-Za-z0-9]+', '_', label).strip('_').lower()


def load_components_for_model_type(model_type):
    if model_type == 'vanilla':
        from train_vanilla_no_T import XPCSDataset, denorm_from_meta, load_model
    elif model_type == 'adv':
        from train_adv_no_T import XPCSDataset, denorm_from_meta, load_model
    elif model_type == 'coral':
        from train_adv_coral_distill import XPCSDataset, denorm_from_meta, load_model
    elif model_type == 'coral-surrogate':
        from train_adv_coral_surrogate import XPCSDataset, denorm_from_meta, load_model
    else:
        raise ValueError(model_type)
    return XPCSDataset, denorm_from_meta, load_model


def phase_noneq_table(sim_root=SIM_ROOT):
    manifest = pd.read_csv(Path(sim_root) / 'manifest.csv')
    noneq_path = Path(sim_root) / 'manifest_with_non_equ.csv'
    if noneq_path.exists() and 'nonequilibrium_measure_raw' not in manifest.columns:
        noneq = pd.read_csv(noneq_path)
        for col in ['nonequilibrium_measure_raw', 'nonequilibrium_measure']:
            if col in noneq.columns and col not in manifest.columns:
                manifest[col] = noneq[col].to_numpy() if len(noneq) == len(manifest) else np.nan
    noneq_col = 'nonequilibrium_measure_raw' if 'nonequilibrium_measure_raw' in manifest.columns else 'nonequilibrium_measure'
    return manifest[['gamma', 'D', 'GB_conc', noneq_col]].rename(columns={noneq_col: 'S_noneq_raw'}).dropna().query('D > 0').copy()


def interpolate_noneq_from_params(predictions, sim_root=SIM_ROOT):
    table = phase_noneq_table(sim_root)
    points = np.column_stack([table['gamma'].to_numpy(float), np.log10(table['D'].to_numpy(float)), table['GB_conc'].to_numpy(float)])
    values = table['S_noneq_raw'].to_numpy(float)
    query_D = np.clip(predictions['D_pred'].to_numpy(float), table['D'].min(), table['D'].max())
    query = np.column_stack([predictions['gamma_pred'].to_numpy(float), np.log10(query_D), predictions['GB_conc_pred'].to_numpy(float)])
    linear = griddata(points, values, query, method='linear', rescale=True)
    nearest = griddata(points, values, query, method='nearest', rescale=True)
    return np.where(np.isfinite(linear), linear, nearest)


def compute_sim_predictions_for_model(label, model_type, model_path, output_csv, limit=None, sim_root=None, noneq_sim_root=None):
    XPCSDataset, denorm_from_meta, load_model = load_components_for_model_type(model_type)
    sim_root = Path(sim_root) if sim_root is not None else SIM_ROOT
    noneq_sim_root = Path(noneq_sim_root) if noneq_sim_root is not None else SIM_ROOT
    dataset = XPCSDataset(sim_root)
    indices = np.arange(len(dataset)) if limit is None else np.linspace(0, len(dataset) - 1, min(limit, len(dataset)), dtype=int)
    model = load_model(Path(model_path), DEVICE).to(DEVICE).eval()
    rows = []
    with torch.no_grad():
        for index in indices:
            sample = dataset[int(index)]
            x, _, y_raw, T = sample[:4]
            pred_norm = model(x.unsqueeze(0).to(DEVICE), T.unsqueeze(0).to(DEVICE))
            pred_raw = denorm_from_meta(pred_norm.squeeze(0), dataset.norm_meta, device=DEVICE).detach().cpu().numpy()
            rows.append({
                'dataset_index': int(index),
                'gamma_true': float(y_raw[0]), 'D_true': float(y_raw[1]), 'GB_conc_true': float(y_raw[2]),
                'gamma_pred': float(pred_raw[0]), 'D_pred': float(pred_raw[1]), 'GB_conc_pred': float(pred_raw[2]),
                'S_noneq_true': float(dataset.manifest.iloc[int(index)].get('nonequilibrium_measure_raw', dataset.manifest.iloc[int(index)].get('nonequilibrium_measure', np.nan))),
            })
    frame = pd.DataFrame(rows)
    frame['S_noneq_pred'] = interpolate_noneq_from_params(frame, noneq_sim_root)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_csv, index=False)
    return frame


def load_or_compute_predictions(spec, output_dir=ML_OUTPUT_DIR / 'sim_predictions'):
    output_dir.mkdir(parents=True, exist_ok=True)
    pred_path = Path(spec.get('sim_predictions') or output_dir / f"{slugify(spec['label'])}.csv")
    if pred_path.exists():
        frame = pd.read_csv(pred_path)
    else:
        frame = compute_sim_predictions_for_model(spec['label'], spec['model_type'], spec['model_path'], pred_path)
    return frame


def plot_pred_vs_true(frame, quantity, output_path, color='#2f6fbb', marker='o'):
    specs = {
        'D': {'true': 'D_true', 'pred': 'D_pred', 'scale': 1e22, 'xlabel': r'Real $D$ ($10^{-22}$ cm$^2\cdot$s$^{-1}$)', 'ylabel': r'Predicted $D$ ($10^{-22}$ cm$^2\cdot$s$^{-1}$)'},
        'GB_conc': {'true': 'GB_conc_true', 'pred': 'GB_conc_pred', 'scale': 1.0, 'xlabel': r'Real $\lambda_{\mathrm{GB}}$', 'ylabel': r'Predicted $\lambda_{\mathrm{GB}}$'},
    }
    spec = specs[quantity]
    x = frame[spec['true']].to_numpy(float) * spec['scale']
    y = frame[spec['pred']].to_numpy(float) * spec['scale']
    finite = np.isfinite(x) & np.isfinite(y)
    x, y = x[finite], y[finite]
    lo = min(float(np.nanmin(x)), float(np.nanmin(y)))
    hi = max(float(np.nanmax(x)), float(np.nanmax(y)))
    pad = 0.04 * (hi - lo if hi > lo else 1.0)
    fig, ax = plt.subplots(figsize=(7.3, 7))
    ax.scatter(x, y, s=15, color=color, marker=marker, alpha=0.42, edgecolors='none')
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color='0.20', lw=1.0, ls='--')
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel(spec['xlabel'])
    ax.set_ylabel(spec['ylabel'])
    if quantity == 'D':
        ax.set_xticks([0, 5, 10])
        ax.set_yticks([0, 5, 10])
    ax.text(0.04, 0.95, rf'$R^2 = {r2_score_np(x, y):.3f}$', transform=ax.transAxes, ha='left', va='top', fontsize=24, bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#d1d5db', lw=1.0, alpha=0.96))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches='tight')
    display(fig)
    plt.close(fig)


def plot_phase_overlay_for_results(results_dir, model_name, output_path, title=None, x_range=(3e-24, 3e-21), y_range=(0.0, 0.5), marker_size=230, dpi=144):
    overlay_df = load_overlay_points(Path(results_dir), model_name=model_name, aggregate_by='shot', shot_index=0)
    overlay_df = add_overlay_metadata(overlay_df, phase_background_df)
    grid_x, grid_y, grid_z = interpolate_phase_grid(phase_background_df, x_range=x_range, y_range=y_range)
    fig, ax = plt.subplots(figsize=(8, 7.0))
    contour = ax.contourf(grid_x, grid_y, grid_z, levels=np.linspace(0, 1, 101), cmap='plasma', norm=Normalize(vmin=0, vmax=1))
    for collection in getattr(contour, 'collections', []):
        collection.set_rasterized(True)
    visible = overlay_df['D'].between(x_range[0], x_range[1]) & overlay_df['GB_conc'].between(y_range[0], y_range[1])
    visible_df = overlay_df.loc[visible].copy()
    for _, style in visible_df[['temperature_c', 'temperature_order', 'temperature_label', 'temperature_color', 'temperature_marker']].drop_duplicates().sort_values(['temperature_order', 'temperature_c']).iterrows():
        group = visible_df.loc[visible_df['temperature_c'] == style['temperature_c']]
        ax.scatter(group['D'], group['GB_conc'], c=group['experimental_nonequilibrium_measure'], cmap='plasma', norm=Normalize(vmin=0, vmax=1), s=marker_size, marker=str(style['temperature_marker']), edgecolors=str(style['temperature_color']), linewidths=1.7, alpha=0.95, zorder=4)
    ax.set_xscale('log')
    ax.set_xlim(*x_range)
    ax.set_ylim(*y_range)
    ax.set_xlabel(r'Diffusivity $D$ (cm$^2\cdot$s$^{-1}$)')
    ax.set_ylabel(r'Effective GB concentration $\lambda_{\mathrm{GB}}$')
    ax.set_xticks([1e-23, 1e-22, 1e-21], [r'$10^{-23}$', r'$10^{-22}$', r'$10^{-21}$'])
    ax.set_yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5], ['0', '0.1', '0.2', '0.3', '0.4', '0.5'])
    if title:
        ax.set_title(title, pad=10)
    cbar = fig.colorbar(contour, ax=ax, location='right', pad=0.05)
    cbar.set_label('Non-equilibrium measure')
    cbar.set_ticks([0.0, 0.5, 1.0])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches='tight', dpi=dpi)
    display(fig)
    plt.close(fig)
    return overlay_df


def raw_files_from_result_dir(result_dir):
    file_names = []
    for csv_path in sorted(Path(result_dir).glob('*/*.csv')):
        if csv_path.parent.name.startswith('phase_diagrams') or csv_path.name.startswith('material_finetune'):
            continue
        try:
            frame = pd.read_csv(csv_path, usecols=['file_name'])
        except Exception:
            continue
        if not frame.empty:
            file_names.append(str(frame['file_name'].iloc[0]))
    return [EXP_RAW_DIR / name for name in sorted(set(file_names))]


def extract_umap_features(model, dataset, batch_size=32, num_workers=2):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    model.eval().to(DEVICE)
    features = []
    with torch.no_grad():
        for batch in loader:
            x = batch[0].to(DEVICE)
            T = batch[3].to(DEVICE)
            if hasattr(model, 'extract_features') and hasattr(model, 'build_shared_features'):
                xpcs_features, temp_features = model.extract_features(x, T)
                latent = model.build_shared_features(xpcs_features, temp_features)
            else:
                latent = model.conv_net(x)
            features.append(latent.detach().cpu())
    return torch.cat(features, dim=0).numpy()


def compute_umap_embedding_for_model(spec, sim_limit=None):
    if not HAS_UMAP:
        raise RuntimeError(f'UMAP is unavailable: {UMAP_IMPORT_ERROR!r}')
    XPCSDataset, _, load_model = load_components_for_model_type(spec['model_type'])
    sim_dataset = XPCSDataset(Path(spec.get('sim_root', SIM_ROOT)))
    if sim_limit is not None and 0 < sim_limit < len(sim_dataset):
        sim_indices = np.linspace(0, len(sim_dataset) - 1, sim_limit, dtype=int).tolist()
        sim_dataset = Subset(sim_dataset, sim_indices)
    raw_files = raw_files_from_result_dir(spec['result_dir'])
    transform_spec = spec.get('input_transform_spec')
    if transform_spec is not None and 'prepare_control_experiment_shots' in globals():
        samples = prepare_control_experiment_shots(
            raw_files=raw_files,
            transform_spec=transform_spec,
            shot_indices=None,
            crop_size=2500,
            coarse_size=256,
            crop_step=100,
            crop_policy='top-left',
            no_t=True,
        )
    else:
        samples = prepare_experiment_shots(raw_files=raw_files, shot_indices=None, crop_size=2500, coarse_size=256, crop_step=100, crop_policy='top-left', no_t=True, cache_dir=PROJECT_ROOT / 'dataset' / 'experiment_eval_cache')
    exp_dataset = PreparedExperimentDataset(samples)
    model = load_model(Path(spec['model_path']), DEVICE)
    sim_features = extract_umap_features(model, sim_dataset)
    exp_features = extract_umap_features(model, exp_dataset)
    all_features = np.vstack([sim_features, exp_features])
    domain_labels = np.concatenate([np.zeros(sim_features.shape[0], dtype=int), np.ones(exp_features.shape[0], dtype=int)])
    reducer = umap.UMAP(n_components=2, n_neighbors=max(2, min(5, all_features.shape[0] - 1)), min_dist=0.1, metric='euclidean', init='spectral', random_state=42)
    coords = reducer.fit_transform(all_features)
    frame = pd.DataFrame({'umap_1': coords[:, 0], 'umap_2': coords[:, 1], 'domain': np.where(domain_labels == 0, 'simulation', 'experiment')})
    return frame


def plot_umap_for_model(spec, output_path, sim_limit=None):
    frame = compute_umap_embedding_for_model(spec, sim_limit=sim_limit)
    fig, ax = plt.subplots(figsize=(8, 6))
    sim_mask = frame['domain'].to_numpy() == 'simulation'
    exp_mask = frame['domain'].to_numpy() == 'experiment'
    ax.scatter(frame.loc[sim_mask, 'umap_1'], frame.loc[sim_mask, 'umap_2'], s=spec.get('sim_marker_size', 400), marker=spec.get('sim_marker', 'o'), color='#4DBBD5FF', alpha=1.0, edgecolors='none', label='Simulation')
    ax.scatter(frame.loc[exp_mask, 'umap_1'], frame.loc[exp_mask, 'umap_2'], s=spec.get('exp_marker_size', 400), marker=spec.get('exp_marker', 'o'), color='#F39B7FFF', alpha=1.0, edgecolors='none', label='Experiment')
    x_min, x_max = frame['umap_1'].min(), frame['umap_1'].max()
    y_min, y_max = frame['umap_2'].min(), frame['umap_2'].max()
    ax.set_xlim(x_min - 0.10 * (x_max - x_min), x_max + 0.10 * (x_max - x_min))
    ax.set_ylim(y_min - 0.10 * (y_max - y_min), y_max + 0.10 * (y_max - y_min))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    csv_path = output_path.with_suffix('.csv')
    frame.to_csv(csv_path, index=False)
    return frame


## 3 And 4. Existing ML Ablations: InfoNCE And Min-Max Adversarial

These cells generate the same source families requested for ML sections: separate `R^2` scatter plots for `D` and `lambda_GB`, a `lambda_GB` vs `D` phase overlay PNG, and UMAP PDF.


In [ ]:
ML_MODEL_SPECS = [
    {
        'label': 'InfoNCE',
        'model_type': 'coral',
        'model_path': PROJECT_ROOT / 'models' / 'XPCS_coral_no_T_best_20260422-003632.pt',
        'result_dir': PROJECT_ROOT / 'results' / 'coral_noT_contrastive-soft-infonce_cw1_top-left_20260422-004405',
        'phase_model_name': 'adv',
        'color': '#6A51A3',
        'marker': 'o',
        'sim_marker': 'o',
        'exp_marker': 'o',
        'sim_predictions': ML_OUTPUT_DIR / 'sim_predictions' / 'infonce.csv',
    },
    {
        # This adversarial run keeps non-constant experiment coordinates better than 20260414-210333.
        'label': 'Adversarial min-max',
        'model_type': 'adv',
        'model_path': PROJECT_ROOT / 'models' / 'XPCS_no_T_best_20260414-150822.pt',
        'result_dir': PROJECT_ROOT / 'results' / 'adv_noT_advseed42_all-diagonal_20260414-152448',
        'phase_model_name': 'adv',
        'color': '#7A7A7A',
        'marker': '^',
        'sim_marker': '^',
        'exp_marker': '^',
        'sim_marker_size': 400,
        'exp_marker_size': 400,
        'sim_predictions': ML_OUTPUT_DIR / 'sim_predictions' / 'adversarial_min_max_20260414_150822.csv',
    },
]

adv_candidate_rows = []
for result_dir in sorted((PROJECT_ROOT / 'results').glob('adv_noT_advseed42_all-diagonal_*')):
    try:
        overlay = load_overlay_points(result_dir, 'adv', aggregate_by='shot', shot_index=0)
    except Exception:
        continue
    if overlay.empty:
        continue
    adv_candidate_rows.append({
        'result_dir': str(result_dir),
        'n': len(overlay),
        'D_std': overlay['D'].std(),
        'GB_conc_std': overlay['GB_conc'].std(),
        'D_median': overlay['D'].median(),
        'GB_conc_median': overlay['GB_conc'].median(),
        'D_min': overlay['D'].min(),
        'D_max': overlay['D'].max(),
        'GB_conc_min': overlay['GB_conc'].min(),
        'GB_conc_max': overlay['GB_conc'].max(),
    })
adv_candidate_table = pd.DataFrame(adv_candidate_rows).sort_values(['GB_conc_std', 'D_std'], ascending=False)
adv_candidate_table.to_csv(ML_OUTPUT_DIR / 'adversarial_result_candidate_scan.csv', index=False)
adv_candidate_table.head(12)


In [ ]:
if RUN_EXISTING_ML_EXPORT:
    ml_summary_rows = []
    for spec in ML_MODEL_SPECS:
        model_dir = ML_OUTPUT_DIR / slugify(spec['label'])
        model_dir.mkdir(parents=True, exist_ok=True)
        pred_frame = load_or_compute_predictions(spec)
        for quantity in ['D', 'GB_conc']:
            plot_pred_vs_true(pred_frame, quantity, model_dir / f'{slugify(spec["label"])}_pred_vs_true_{quantity}.pdf', color=spec['color'], marker=spec['marker'])
            qspec = {'D': ('D_true', 'D_pred', 1e22), 'GB_conc': ('GB_conc_true', 'GB_conc_pred', 1.0)}[quantity]
            ml_summary_rows.append({'model': spec['label'], 'quantity': quantity, 'R2': r2_score_np(pred_frame[qspec[0]].to_numpy(float) * qspec[2], pred_frame[qspec[1]].to_numpy(float) * qspec[2]), 'n': len(pred_frame)})
        overlay = plot_phase_overlay_for_results(spec['result_dir'], spec['phase_model_name'], model_dir / f'{slugify(spec["label"])}_phase_D_GB.png', title=None)
        overlay.to_csv(model_dir / f'{slugify(spec["label"])}_overlay_points.csv', index=False)
        if RUN_UMAP_EXPORT:
            plot_umap_for_model(spec, model_dir / f'{slugify(spec["label"])}_UMAP.pdf', sim_limit=None)
    ml_summary = pd.DataFrame(ml_summary_rows)
    ml_summary.to_csv(ML_OUTPUT_DIR / 'existing_ml_summary.csv', index=False)
    display(ml_summary)
else:
    print('Existing ML export disabled.')


## 3. Smearing And Rescaling Vanilla Input Controls

This section creates transformed simulation datasets, trains ordinary vanilla no-T regressors on those transformed simulations, and evaluates experimental inputs after the same deterministic transform. No CORAL alignment or non-equilibrium surrogate loss is used in this control.


In [ ]:
AUG_OUTPUT_DIR = FIGURE_ROOT / 'FigS_smearing_rescaling_controls'
AUG_RESULT_DIR = RESULT_ROOT / 'input_transform_vanilla_controls'
AUG_DATASET_DIR = PROJECT_ROOT / 'dataset' / 'supplementary_rewrite_input_controls'
AUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUG_RESULT_DIR.mkdir(parents=True, exist_ok=True)
AUG_DATASET_DIR.mkdir(parents=True, exist_ok=True)

VANILLA_REFERENCE_JSON = PROJECT_ROOT / 'models' / 'Vanilla_XPCS_no_T_best_20260414-202028.json'
with open(VANILLA_REFERENCE_JSON) as f:
    vanilla_reference_config = json.load(f)

AUG_TRAIN_CONFIG = {
    'epochs': int(vanilla_reference_config.get('epochs', 100)),
    'batch_size': int(vanilla_reference_config.get('batch_size', 32)),
    'learning_rate': float(vanilla_reference_config.get('learning_rate', 1e-3)),
    'seed': int(vanilla_reference_config.get('seed', 42)),
    'num_workers': 0,
}

AUGMENTATION_SPECS = [
    {'label': 'smearing', 'kind': 'smear', 'kernel': 5, 'alpha': 0.35},
    {'label': 'rescaling', 'kind': 'rescale', 'low': 1.0, 'high': 1.2},
]

AUG_EVAL_CONFIG = {
    'crop_size': 2500,
    'coarse_size': 256,
    'crop_step': 100,
    'crop_policy': 'all-diagonal',
    'crop_aggregation': 'mean',
    'eval_batch_size': 32,
    'shot_indices': [0],
}
AUG_TRAIN_CONFIG


In [ ]:
from produce_data import coarse_grain_g2
from run_all import ShotSample, build_diag_mask, build_diagonal_crop_starts, load_raw_experiment_array, parse_temperature_from_name
from train_vanilla_no_T import INPUT_MEAN as INPUT_MEAN_NO_T, INPUT_STD as INPUT_STD_NO_T
from utils import nonequilibrium_measure


def apply_input_transform_tensor(x, spec):
    x = x.to(torch.float32).clone()
    if x.ndim == 2:
        x2 = x
    elif x.ndim == 3 and x.shape[0] == 1:
        x2 = x.squeeze(0)
    else:
        x2 = x.squeeze()
    if spec['kind'] == 'smear':
        kernel = int(spec.get('kernel', 5))
        if kernel % 2 == 0:
            kernel += 1
        alpha = float(spec.get('alpha', 0.35))
        pad = kernel // 2
        blurred = F.avg_pool2d(
            F.pad(x2[None, None], (pad, pad, pad, pad), mode='reflect'),
            kernel_size=kernel,
            stride=1,
        ).squeeze(0).squeeze(0)
        x2 = (1.0 - alpha) * x2 + alpha * blurred
    elif spec['kind'] == 'rescale':
        x_np = normalize_g2_report(
            x2,
            min_val=float(spec.get('low', 1.0)),
            max_val=float(spec.get('high', 1.2)),
            diagonal_value=float(spec.get('high', 1.2)),
            ignore_diagonal=True,
        )
        x2 = torch.as_tensor(x_np, dtype=torch.float32)
    else:
        raise ValueError(spec['kind'])
    return x2.unsqueeze(0)


def make_transformed_simulation_dataset(spec, source_root=SIM_ROOT, force=False):
    slug = slugify(spec['label'])
    dataset_root = AUG_DATASET_DIR / f'simulation_{slug}'
    tensor_dir = dataset_root / 'tensors'
    manifest_path = dataset_root / 'manifest.csv'
    if manifest_path.exists() and not force:
        return dataset_root
    if dataset_root.exists() and force:
        shutil.rmtree(dataset_root)
    tensor_dir.mkdir(parents=True, exist_ok=True)
    manifest = pd.read_csv(Path(source_root) / 'manifest.csv')
    rows = []
    for output_index, (_, row) in enumerate(manifest.iterrows()):
        source_path = PROJECT_ROOT / str(row['path']) if not Path(str(row['path'])).is_absolute() else Path(str(row['path']))
        x = torch.load(source_path, map_location='cpu', weights_only=True)
        transformed = apply_input_transform_tensor(x, spec)
        out_path = tensor_dir / f'{output_index:06d}.pt'
        torch.save(transformed, out_path)
        out_row = row.copy()
        out_row['source_sim_path'] = row['path']
        out_row['source_sim_id'] = row.get('id', output_index)
        if 'nonequilibrium_measure_raw' in out_row.index:
            out_row['source_nonequilibrium_measure_raw'] = out_row['nonequilibrium_measure_raw']
        if 'nonequilibrium_measure' in out_row.index:
            out_row['source_nonequilibrium_measure'] = out_row['nonequilibrium_measure']
        out_row['path'] = str(out_path.relative_to(PROJECT_ROOT))
        out_row['domain'] = 'simulation'
        out_row['id'] = output_index
        out_row['nonequilibrium_measure_raw'] = nonequilibrium_measure(transformed)
        out_row['nonequilibrium_measure'] = out_row['nonequilibrium_measure_raw']
        rows.append(out_row)
    transformed_manifest = pd.DataFrame(rows)
    transformed_manifest.to_csv(manifest_path, index=False)
    keep = [col for col in ['id', 'gamma', 'D', 'GB_conc', 'T', 'path', 'domain', 'nonequilibrium_measure_raw', 'nonequilibrium_measure'] if col in transformed_manifest.columns]
    transformed_manifest[keep].to_csv(dataset_root / 'manifest_with_non_equ.csv', index=False)
    (dataset_root / 'transform_config.json').write_text(json.dumps(spec, indent=2), encoding='ascii')
    return dataset_root


def prepare_control_experiment_shots(
    raw_files,
    transform_spec,
    shot_indices=None,
    crop_size=2500,
    coarse_size=256,
    crop_step=100,
    crop_policy='all-diagonal',
    no_t=True,
):
    if not no_t:
        raise ValueError('The SI input controls use the no-T model path.')
    diag_mask = build_diag_mask(coarse_size)
    selected_indices = None if shot_indices is None else set(shot_indices)
    samples = []
    for raw_file in raw_files:
        data = load_raw_experiment_array(Path(raw_file))
        temperature = parse_temperature_from_name(Path(raw_file))
        crop_starts = build_diagonal_crop_starts(data.shape[:2], crop_size, crop_step, crop_policy)
        n_shots = data.shape[-1]
        selected_shots = list(range(n_shots)) if selected_indices is None else [idx for idx in sorted(selected_indices) if 0 <= idx < n_shots]
        for shot_index in selected_shots:
            for crop_start in crop_starts:
                shot = torch.tensor(
                    data[crop_start:crop_start + crop_size, crop_start:crop_start + crop_size, shot_index],
                    dtype=torch.float32,
                )
                g2_raw = coarse_grain_g2(shot, target_size=(coarse_size, coarse_size)).to(torch.float32)
                g2_raw = apply_input_transform_tensor(g2_raw, transform_spec).squeeze(0)
                measured_noneq = nonequilibrium_measure(g2_raw)
                g2_input = (g2_raw - INPUT_MEAN_NO_T) / (INPUT_STD_NO_T + 1e-6)
                samples.append(
                    ShotSample(
                        file_name=Path(raw_file).name,
                        file_stem=Path(raw_file).stem,
                        shot_index=shot_index,
                        crop_start=crop_start,
                        temperature_k=temperature,
                        x=g2_input.unsqueeze(0) * diag_mask,
                        nonequilibrium_measure=measured_noneq,
                    )
                )
        print(f"[evaluate] prepared {Path(raw_file).name} with {len(selected_shots)} selected shot(s), {len(crop_starts)} crop(s), transform={transform_spec['label']}")
    if not samples:
        raise RuntimeError('No experiment shots were selected for evaluation.')
    return samples


def train_vanilla_input_control(sim_root, run_root):
    from train_vanilla_no_T import VanillaXPCSNet, train as train_vanilla
    run_root = Path(run_root)
    existing = sorted((run_root / 'models').glob('Vanilla_XPCS_no_T_best_*.pt'), key=lambda p: p.stat().st_mtime)
    if existing:
        print(f'[train] reusing existing vanilla checkpoint for {run_root.name}: {existing[-1]}')
        return None, existing[-1]
    cfg = dict(AUG_TRAIN_CONFIG)
    model = VanillaXPCSNet(predictor_output_activation='sigmoid')
    trained = train_vanilla(
        model,
        sim_root=Path(sim_root),
        batch_size=cfg['batch_size'],
        epochs=cfg['epochs'],
        learning_rate=cfg['learning_rate'],
        seed=cfg['seed'],
        deterministic=True,
        num_workers=cfg['num_workers'],
        device=DEVICE,
        log_pardir=Path(run_root) / 'runs',
        model_path=Path(run_root) / 'models',
    )
    checkpoint = sorted((Path(run_root) / 'models').glob('Vanilla_XPCS_no_T_best_*.pt'), key=lambda p: p.stat().st_mtime)[-1]
    return trained, checkpoint


def experiment_result_csvs(results_dir):
    results_dir = Path(results_dir)
    if not results_dir.exists():
        return []
    return [p for p in sorted(results_dir.glob('*/*.csv')) if not p.parent.name.startswith('phase_diagrams') and not p.name.startswith('material_finetune')]


def run_experiment_inference_for_checkpoint(label, model_type, model_path, result_dir, files=None, force=False, transform_spec=None):
    result_dir = Path(result_dir)
    if force and result_dir.exists():
        shutil.rmtree(result_dir)
    if experiment_result_csvs(result_dir) and not force:
        return result_dir
    XPCSDataset, denorm_from_meta, load_model = load_components_for_model_type(model_type)
    del XPCSDataset
    raw_files = filter_files(iter_raw_experiment_files(EXP_RAW_DIR), files)
    if not raw_files:
        raise RuntimeError(f'No raw experiment files selected for {label}.')
    if transform_spec is None:
        samples = prepare_experiment_shots(
            raw_files=raw_files,
            shot_indices=AUG_EVAL_CONFIG['shot_indices'],
            crop_size=AUG_EVAL_CONFIG['crop_size'],
            coarse_size=AUG_EVAL_CONFIG['coarse_size'],
            crop_step=AUG_EVAL_CONFIG['crop_step'],
            crop_policy=AUG_EVAL_CONFIG['crop_policy'],
            no_t=True,
            cache_dir=PROJECT_ROOT / 'dataset' / 'experiment_eval_cache',
        )
    else:
        samples = prepare_control_experiment_shots(
            raw_files=raw_files,
            transform_spec=transform_spec,
            shot_indices=AUG_EVAL_CONFIG['shot_indices'],
            crop_size=AUG_EVAL_CONFIG['crop_size'],
            coarse_size=AUG_EVAL_CONFIG['coarse_size'],
            crop_step=AUG_EVAL_CONFIG['crop_step'],
            crop_policy=AUG_EVAL_CONFIG['crop_policy'],
            no_t=True,
        )
    dataset = PreparedExperimentDataset(samples)
    model = load_model(Path(model_path), DEVICE)
    predictions = predict_samples(
        model=model,
        dataset=dataset,
        batch_size=AUG_EVAL_CONFIG['eval_batch_size'],
        device=DEVICE,
        denorm_fn=denorm_from_meta,
    )
    write_kwargs = {
        'samples': samples,
        'adv_predictions': None,
        'vanilla_predictions': None,
        'adv_coords': None,
        'adv_sim_count': 0,
        'vanilla_coords': None,
        'vanilla_sim_count': 0,
        'results_dir': result_dir,
        'skip_umap': True,
        'crop_policy': AUG_EVAL_CONFIG['crop_policy'],
        'crop_aggregation': AUG_EVAL_CONFIG['crop_aggregation'],
    }
    if model_type == 'vanilla':
        write_kwargs['vanilla_predictions'] = predictions
    else:
        write_kwargs['adv_predictions'] = predictions
    write_results_csvs(**write_kwargs)
    return result_dir


def _draw_pred_vs_true(ax, frame, quantity, color='#2f6fbb', marker='o'):
    specs = {
        'D': {'true': 'D_true', 'pred': 'D_pred', 'scale': 1e22, 'xlabel': r'Real $D$ ($10^{-22}$ cm$^2\cdot$s$^{-1}$)', 'ylabel': r'Predicted $D$ ($10^{-22}$ cm$^2\cdot$s$^{-1}$)'},
        'GB_conc': {'true': 'GB_conc_true', 'pred': 'GB_conc_pred', 'scale': 1.0, 'xlabel': r'Real $\lambda_{\mathrm{GB}}$', 'ylabel': r'Predicted $\lambda_{\mathrm{GB}}$'},
    }
    spec = specs[quantity]
    x = frame[spec['true']].to_numpy(float) * spec['scale']
    y = frame[spec['pred']].to_numpy(float) * spec['scale']
    finite = np.isfinite(x) & np.isfinite(y)
    x, y = x[finite], y[finite]
    lo = min(float(np.nanmin(x)), float(np.nanmin(y)))
    hi = max(float(np.nanmax(x)), float(np.nanmax(y)))
    pad = 0.04 * (hi - lo if hi > lo else 1.0)
    ax.scatter(x, y, s=7, color=color, marker=marker, alpha=0.42, edgecolors='none')
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color='0.20', lw=0.9, ls='--')
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_xlabel(spec['xlabel'])
    ax.set_ylabel(spec['ylabel'])
    if quantity == 'D':
        ax.set_xticks([0, 5, 10])
        ax.set_yticks([0, 5, 10])
    ax.text(0.04, 0.95, rf'$R^2 = {r2_score_np(x, y):.3f}$', transform=ax.transAxes, ha='left', va='top', fontsize=16, bbox=dict(boxstyle='round,pad=0.28', fc='white', ec='#d1d5db', lw=0.8, alpha=0.96))


def _draw_phase_overlay(ax, overlay_df, x_range=(3e-24, 3e-21), y_range=(0.0, 0.5), marker_size=58):
    grid_x, grid_y, grid_z = interpolate_phase_grid(phase_background_df, x_range=x_range, y_range=y_range)
    contour = ax.contourf(grid_x, grid_y, grid_z, levels=np.linspace(0, 1, 101), cmap='plasma', norm=Normalize(vmin=0, vmax=1))
    for collection in getattr(contour, 'collections', []):
        collection.set_rasterized(True)
    visible = overlay_df['D'].between(x_range[0], x_range[1]) & overlay_df['GB_conc'].between(y_range[0], y_range[1])
    visible_df = overlay_df.loc[visible].copy()
    for _, style in visible_df[['temperature_c', 'temperature_order', 'temperature_color', 'temperature_marker']].drop_duplicates().sort_values(['temperature_order', 'temperature_c']).iterrows():
        group = visible_df.loc[visible_df['temperature_c'] == style['temperature_c']]
        ax.scatter(group['D'], group['GB_conc'], c=group['experimental_nonequilibrium_measure'], cmap='plasma', norm=Normalize(vmin=0, vmax=1), s=marker_size, marker=str(style['temperature_marker']), edgecolors=str(style['temperature_color']), linewidths=0.9, alpha=0.95, zorder=4)
    ax.set_xscale('log')
    ax.set_xlim(*x_range)
    ax.set_ylim(*y_range)
    ax.set_xlabel(r'Diffusivity $D$ (cm$^2\cdot$s$^{-1}$)')
    ax.set_ylabel(r'Effective GB concentration $\lambda_{\mathrm{GB}}$')
    ax.set_xticks([1e-23, 1e-22, 1e-21], [r'$10^{-23}$', r'$10^{-22}$', r'$10^{-21}$'])
    ax.set_yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5], ['0', '0.1', '0.2', '0.3', '0.4', '0.5'])
    return contour


def _draw_umap(ax, umap_frame, spec):
    sim_mask = umap_frame['domain'].to_numpy() == 'simulation'
    exp_mask = umap_frame['domain'].to_numpy() == 'experiment'
    ax.scatter(umap_frame.loc[sim_mask, 'umap_1'], umap_frame.loc[sim_mask, 'umap_2'], s=70, marker=spec.get('sim_marker', 'o'), color='#4DBBD5FF', alpha=1.0, edgecolors='none', label='Theoretical XPCS')
    ax.scatter(umap_frame.loc[exp_mask, 'umap_1'], umap_frame.loc[exp_mask, 'umap_2'], s=76, marker=spec.get('exp_marker', 'o'), color='#F39B7FFF', alpha=1.0, edgecolors='none', label='Experimental XPCS')
    x_min, x_max = umap_frame['umap_1'].min(), umap_frame['umap_1'].max()
    y_min, y_max = umap_frame['umap_2'].min(), umap_frame['umap_2'].max()
    ax.set_xlim(x_min - 0.10 * (x_max - x_min), x_max + 0.10 * (x_max - x_min))
    ax.set_ylim(y_min - 0.10 * (y_max - y_min), y_max + 0.10 * (y_max - y_min))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, handletextpad=0.35, columnspacing=0.8, fontsize=13)


def export_control_composite_figure(label, pred_frame, overlay_df, umap_frame, spec, output_path):
    with mpl.rc_context({
        'font.size': 16,
        'axes.labelsize': 18,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'axes.linewidth': 1.0,
        'xtick.major.width': 1.0,
        'ytick.major.width': 1.0,
    }):
        fig = plt.figure(figsize=(12.0, 9.2))
        gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.08], width_ratios=[1.0, 1.0], hspace=0.43, wspace=0.34, left=0.08, right=0.96, bottom=0.08, top=0.96)
        top = gs[0, :].subgridspec(1, 2, wspace=0.34)
        ax_d = fig.add_subplot(top[0, 0])
        ax_gb = fig.add_subplot(top[0, 1])
        ax_phase = fig.add_subplot(gs[1, 0])
        ax_umap = fig.add_subplot(gs[1, 1])
        _draw_pred_vs_true(ax_d, pred_frame, 'D', color=spec['color'], marker=spec['marker'])
        _draw_pred_vs_true(ax_gb, pred_frame, 'GB_conc', color=spec['color'], marker=spec['marker'])
        contour = _draw_phase_overlay(ax_phase, overlay_df)
        _draw_umap(ax_umap, umap_frame, spec)
        cbar = fig.colorbar(contour, ax=ax_phase, location='right', pad=0.04, fraction=0.055)
        cbar.set_label('Non-equilibrium measure', fontsize=17)
        cbar.set_ticks([0.0, 0.5, 1.0])
        cbar.ax.tick_params(labelsize=14)
        for letter, ax in [('a', ax_d), ('b', ax_phase), ('c', ax_umap)]:
            ax.text(-0.12, 1.07, letter, transform=ax.transAxes, fontsize=22, fontweight='bold', va='top', ha='left')
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, bbox_inches='tight', pad_inches=0.05)
        display(fig)
        plt.close(fig)


def evaluate_trained_control(label, model_path, result_dir, figure_dir, sim_root, transform_spec):
    spec = {
        'label': label,
        'model_type': 'vanilla',
        'model_path': Path(model_path),
        'result_dir': Path(result_dir) / 'experiment_inference',
        'phase_model_name': 'vanilla',
        'color': '#B84A62',
        'marker': 's',
        'sim_marker': 's',
        'exp_marker': '^',
        'sim_predictions': Path(result_dir) / 'predictions_transformed_sim.csv',
        'sim_root': Path(sim_root),
        'input_transform_spec': dict(transform_spec),
    }
    if not spec['sim_predictions'].exists():
        frame = compute_sim_predictions_for_model(label, spec['model_type'], spec['model_path'], spec['sim_predictions'], sim_root=sim_root, noneq_sim_root=SIM_ROOT)
    else:
        frame = pd.read_csv(spec['sim_predictions'])
    for quantity in ['D', 'GB_conc']:
        plot_pred_vs_true(frame, quantity, Path(figure_dir) / f'{slugify(label)}_pred_vs_true_{quantity}.pdf', color=spec['color'], marker=spec['marker'])
    run_experiment_inference_for_checkpoint(label, spec['model_type'], spec['model_path'], spec['result_dir'], force=True, transform_spec=transform_spec)
    overlay = plot_phase_overlay_for_results(spec['result_dir'], spec['phase_model_name'], Path(figure_dir) / f'{slugify(label)}_phase_D_GB.png', title=None)
    overlay.to_csv(Path(figure_dir) / f'{slugify(label)}_overlay_points.csv', index=False)
    umap_frame = None
    if RUN_UMAP_EXPORT:
        umap_frame = plot_umap_for_model(spec, Path(figure_dir) / f'{slugify(label)}_UMAP.pdf', sim_limit=None)
    if EXPORT_SI_COMPOSITES and umap_frame is not None:
        fig_number = {'smearing': 'FigS8', 'rescaling': 'FigS9'}.get(slugify(label))
        if fig_number and WRITEUP_ROOT is not None:
            export_control_composite_figure(label, frame, overlay, umap_frame, spec, WRITEUP_ROOT / 'Figures' / f'{fig_number}.pdf')
    return spec, frame, overlay

if RUN_AUGMENTATION_RETRAINING:
    augmentation_rows = []
    for spec in AUGMENTATION_SPECS:
        sim_root = make_transformed_simulation_dataset(spec)
        run_root = AUG_RESULT_DIR / slugify(spec['label'])
        run_root.mkdir(parents=True, exist_ok=True)
        trained_model, checkpoint = train_vanilla_input_control(sim_root, run_root)
        figure_dir = AUG_OUTPUT_DIR / slugify(spec['label'])
        trained_spec, pred_frame, overlay = evaluate_trained_control(spec['label'], checkpoint, run_root, figure_dir, sim_root=sim_root, transform_spec=spec)
        augmentation_rows.append({
            'label': spec['label'],
            'sim_root': str(sim_root),
            'checkpoint': str(checkpoint),
            'sim_predictions': str(trained_spec['sim_predictions']),
            'R2_D': r2_score_np(pred_frame['D_true'].to_numpy(float) * 1e22, pred_frame['D_pred'].to_numpy(float) * 1e22),
            'R2_GB_conc': r2_score_np(pred_frame['GB_conc_true'].to_numpy(float), pred_frame['GB_conc_pred'].to_numpy(float)),
            'n_experiment_points': int(len(overlay)),
            'n_lambda_le_0p02': int((overlay['GB_conc'] <= 0.02).sum()),
        })
    augmentation_summary = pd.DataFrame(augmentation_rows)
    augmentation_summary.to_csv(AUG_OUTPUT_DIR / 'augmentation_training_summary.csv', index=False)
    display(augmentation_summary)
else:
    print('Long augmentation retraining disabled. Set RUN_AUGMENTATION_RETRAINING=True and run this section to train smearing/rescaling controls.')


In [ ]:
def redraw_cached_input_control_umaps():
    set_report_style()
    style_specs = {
        'smearing': {'sim_marker': 's', 'exp_marker': '^', 'sim_marker_size': 400, 'exp_marker_size': 400},
        'rescaling': {'sim_marker': 's', 'exp_marker': '^', 'sim_marker_size': 400, 'exp_marker_size': 400},
    }
    for label, spec in style_specs.items():
        source_dir = AUG_OUTPUT_DIR / slugify(label)
        frame = pd.read_csv(source_dir / f'{slugify(label)}_UMAP.csv')
        fig, ax = plt.subplots(figsize=(8, 6))
        sim_mask = frame['domain'].to_numpy() == 'simulation'
        exp_mask = frame['domain'].to_numpy() == 'experiment'
        ax.scatter(frame.loc[sim_mask, 'umap_1'], frame.loc[sim_mask, 'umap_2'], s=spec['sim_marker_size'], marker=spec['sim_marker'], color='#4DBBD5FF', alpha=1.0, edgecolors='none', label='Simulation')
        ax.scatter(frame.loc[exp_mask, 'umap_1'], frame.loc[exp_mask, 'umap_2'], s=spec['exp_marker_size'], marker=spec['exp_marker'], color='#F39B7FFF', alpha=1.0, edgecolors='none', label='Experiment')
        x_min, x_max = frame['umap_1'].min(), frame['umap_1'].max()
        y_min, y_max = frame['umap_2'].min(), frame['umap_2'].max()
        ax.set_xlim(x_min - 0.10 * (x_max - x_min), x_max + 0.10 * (x_max - x_min))
        ax.set_ylim(y_min - 0.10 * (y_max - y_min), y_max + 0.10 * (y_max - y_min))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel('Dimension 1')
        ax.set_ylabel('Dimension 2')
        fig.savefig(source_dir / f'{slugify(label)}_UMAP.pdf', bbox_inches='tight')
        display(fig)
        plt.close(fig)

if RUN_UMAP_EXPORT:
    redraw_cached_input_control_umaps()


## 5. `w_coral` / `w_sur` Sweep

This section retrains the final CORAL + surrogate model under matched training config while varying `w_coral` and `w_sur`. It saves only a table: best loss plus held-out simulation `R^2` for `D` and `lambda_GB`.


In [ ]:
WEIGHT_OUTPUT_DIR = FIGURE_ROOT / 'FigS_weight_sweep'
WEIGHT_RESULT_DIR = RESULT_ROOT / 'weight_sweep'
WEIGHT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHT_RESULT_DIR.mkdir(parents=True, exist_ok=True)

WEIGHT_SWEEP_CONFIGS = [
    {'w_coral': 0.0, 'w_sur': 1.2},
    {'w_coral': 0.25, 'w_sur': 1.2},
    {'w_coral': 0.5, 'w_sur': 1.2},
    {'w_coral': 1.0, 'w_sur': 0.0},
    {'w_coral': 1.0, 'w_sur': 0.3},
    {'w_coral': 1.0, 'w_sur': 0.6},
    {'w_coral': 1.0, 'w_sur': 1.2},
    {'w_coral': 1.0, 'w_sur': 2.0},
    {'w_coral': 2.0, 'w_sur': 1.2},
    {'w_coral': 4.0, 'w_sur': 1.2},
]


def summarize_weight_run(run_root, checkpoint, predictions):
    meta_paths = sorted((Path(run_root) / 'models').glob('XPCS_coral_surrogate_no_T_best_*.json'), key=lambda p: p.stat().st_mtime)
    metadata = json.loads(meta_paths[-1].read_text()) if meta_paths else {}
    return {
        'checkpoint': str(checkpoint),
        'best_selection_score': metadata.get('best_selection_score'),
        'sim_pred_loss': (metadata.get('test_metrics') or {}).get('sim_pred_loss'),
        'R2_D': r2_score_np(predictions['D_true'].to_numpy(float) * 1e22, predictions['D_pred'].to_numpy(float) * 1e22),
        'R2_GB_conc': r2_score_np(predictions['GB_conc_true'].to_numpy(float), predictions['GB_conc_pred'].to_numpy(float)),
    }

if RUN_WEIGHT_SWEEP:
    weight_rows = []
    for config in WEIGHT_SWEEP_CONFIGS:
        label = f"wcoral{config['w_coral']:g}_wsur{config['w_sur']:g}".replace('.', 'p')
        run_root = WEIGHT_RESULT_DIR / label
        run_root.mkdir(parents=True, exist_ok=True)
        trained_model, checkpoint = train_coral_surrogate_control(PROJECT_ROOT / 'dataset' / 'experiment', run_root, coral_weight=config['w_coral'], surrogate_weight=config['w_sur'])
        pred_csv = run_root / 'predictions_clean_sim.csv'
        predictions = compute_sim_predictions_for_model(label, 'coral-surrogate', checkpoint, pred_csv)
        row = {'w_coral': config['w_coral'], 'w_sur': config['w_sur'], **summarize_weight_run(run_root, checkpoint, predictions)}
        weight_rows.append(row)
        pd.DataFrame(weight_rows).to_csv(WEIGHT_OUTPUT_DIR / 'weight_sweep_summary_partial.csv', index=False)
    weight_summary = pd.DataFrame(weight_rows)
    weight_summary.to_csv(WEIGHT_OUTPUT_DIR / 'weight_sweep_summary.csv', index=False)
    display(weight_summary)
else:
    print('Long weight sweep disabled. Set RUN_WEIGHT_SWEEP=True and run this section to train the 10 matched-config variants.')
    pd.DataFrame(WEIGHT_SWEEP_CONFIGS)


## 6. Final-Model Minor Outlier Examples

The final material-wise fine-tuned model is scanned for points pushed toward `lambda_GB = 0`. A zoomed phase diagram is saved as PNG, and the corresponding experimental XPCS maps/autocorrelation panels are exported as PDFs.


In [ ]:
OUTLIER_OUTPUT_DIR = FIGURE_ROOT / 'FigS_final_model_outliers'
OUTLIER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_RESULT_DIR = PROJECT_ROOT / 'results' / 'coral-surrogate-material-ft_noT_shot0_predictor_mse_seed42_all-diagonal_20260424-181439'
FINAL_PHASE_MODEL_NAME = 'adv'
OUTLIER_GB_THRESHOLD = 0.005
OUTLIER_MAX_PANELS = 8


def load_final_overlay():
    overlay = load_overlay_points(FINAL_RESULT_DIR, FINAL_PHASE_MODEL_NAME, aggregate_by='shot', shot_index=0)
    overlay = add_overlay_metadata(overlay, phase_background_df)
    overlay = overlay.sort_values(['GB_conc', 'D']).reset_index(drop=True)
    return overlay


def plot_zoomed_outlier_phase(overlay, outliers, output_path):
    x_lo = max(3e-24, float(np.nanmin(outliers['D'])) / 2.5) if len(outliers) else 3e-24
    x_hi = min(3e-21, float(np.nanmax(outliers['D'])) * 2.5) if len(outliers) else 3e-21
    y_range = (0.0, max(0.035, float(np.nanmax(outliers['GB_conc'])) + 0.015 if len(outliers) else 0.04))
    grid_x, grid_y, grid_z = interpolate_phase_grid(phase_background_df, x_range=(x_lo, x_hi), y_range=y_range)
    fig, ax = plt.subplots(figsize=(9.5, 6.6))
    contour = ax.contourf(grid_x, grid_y, grid_z, levels=np.linspace(0, 1, 101), cmap='plasma', norm=Normalize(vmin=0, vmax=1))
    visible = overlay['D'].between(x_lo, x_hi) & overlay['GB_conc'].between(y_range[0], y_range[1])
    near_df = overlay.loc[visible].copy()
    for _, style in near_df[['temperature_c', 'temperature_order', 'temperature_color', 'temperature_marker']].drop_duplicates().sort_values(['temperature_order', 'temperature_c']).iterrows():
        group = near_df.loc[near_df['temperature_c'] == style['temperature_c']]
        ax.scatter(group['D'], group['GB_conc'], c=group['experimental_nonequilibrium_measure'], cmap='plasma', norm=Normalize(vmin=0, vmax=1), s=145, marker=str(style['temperature_marker']), edgecolors=str(style['temperature_color']), linewidths=1.4, alpha=0.92, zorder=4)
    ax.scatter(outliers['D'], outliers['GB_conc'], c=outliers['experimental_nonequilibrium_measure'], cmap='plasma', norm=Normalize(vmin=0, vmax=1), s=300, marker='o', edgecolors='black', linewidths=1.8, zorder=6)
    ax.set_xscale('log')
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(*y_range)
    phase_xticks = [(5e-24, r'$5\times10^{-24}$'), (1e-23, r'$10^{-23}$'), (2e-23, r'$2\times10^{-23}$')]
    visible_xticks = [(tick, label) for tick, label in phase_xticks if x_lo <= tick <= x_hi]
    if len(visible_xticks) >= 2:
        ax.set_xticks([tick for tick, _ in visible_xticks])
        ax.set_xticklabels([label for _, label in visible_xticks])
        ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel(r'Diffusivity $D$ (cm$^2\cdot$s$^{-1}$)')
    ax.set_ylabel(r'Effective GB concentration $\lambda_{\mathrm{GB}}$')
    cbar = fig.colorbar(contour, ax=ax, location='right', pad=0.05)
    cbar.set_label('Non-equilibrium measure')
    cbar.set_ticks([0.0, 0.5, 1.0])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches='tight', dpi=144)
    display(fig)
    plt.close(fig)

if RUN_FINAL_OUTLIER_EXPORT:
    final_overlay = load_final_overlay()
    outliers = final_overlay.loc[final_overlay['GB_conc'] <= OUTLIER_GB_THRESHOLD].copy()
    if outliers.empty:
        outliers = final_overlay.nsmallest(OUTLIER_MAX_PANELS, 'GB_conc').copy()
    else:
        outliers = outliers.nsmallest(OUTLIER_MAX_PANELS, 'GB_conc').copy()
    final_overlay.to_csv(OUTLIER_OUTPUT_DIR / 'final_overlay_points.csv', index=False)
    outliers.to_csv(OUTLIER_OUTPUT_DIR / 'final_low_lambda_GB_outliers.csv', index=False)
    other_points = final_overlay.drop(index=outliers.index, errors='ignore')
    outlier_summary = pd.DataFrame([
        {
            'group': 'low_lambda_GB_outliers',
            'n': len(outliers),
            'D_median': outliers['D'].median(),
            'GB_conc_median': outliers['GB_conc'].median(),
            'gamma_median': outliers['gamma'].median(),
            'experimental_noneq_mean': outliers['experimental_nonequilibrium_measure'].mean(),
            'within_simulation_domain_fraction': outliers['within_simulation_domain'].mean() if 'within_simulation_domain' in outliers else np.nan,
        },
        {
            'group': 'remaining_experiment_points',
            'n': len(other_points),
            'D_median': other_points['D'].median(),
            'GB_conc_median': other_points['GB_conc'].median(),
            'gamma_median': other_points['gamma'].median(),
            'experimental_noneq_mean': other_points['experimental_nonequilibrium_measure'].mean(),
            'within_simulation_domain_fraction': other_points['within_simulation_domain'].mean() if 'within_simulation_domain' in other_points else np.nan,
        },
    ])
    outlier_summary.to_csv(OUTLIER_OUTPUT_DIR / 'final_low_lambda_GB_outlier_summary.csv', index=False)
    display(outlier_summary)
    display(outliers[['material_subdir', 'file_name', 'shot_index', 'temperature_c', 'D', 'GB_conc', 'gamma', 'experimental_nonequilibrium_measure']])
    plot_zoomed_outlier_phase(final_overlay, outliers, OUTLIER_OUTPUT_DIR / 'final_model_low_lambda_GB_zoom_phase.png')

    for obsolete in ['final_low_lambda_GB_outlier_g2.pdf', 'final_low_lambda_GB_outlier_autocorr.pdf']:
        (OUTLIER_OUTPUT_DIR / obsolete).unlink(missing_ok=True)
    for rank, (_, row) in enumerate(outliers.iterrows(), start=1):
        raw_path = EXP_RAW_DIR / str(row['file_name'])
        shot_index = int(row.get('shot_index', 0)) if np.isfinite(row.get('shot_index', 0)) else 0
        g2 = load_experiment_panel(raw_path, channel=shot_index, t_max_s=2500, coarse_size=256)
        panel = {
            'label': '',
            'g2': g2,
            'file_name': row['file_name'],
            'GB_conc': row['GB_conc'],
        }
        safe_name = re.sub(r'[^A-Za-z0-9_.-]+', '_', f"{row['material_subdir']}_{row['temperature_c']:.0f}C_shot{shot_index}")
        print(f"outlier {rank:02d}: {row['material_subdir']} | {row['temperature_c']:.0f} C | {row['file_name']} | shot {shot_index} | D={row['D']:.3e} | lambda_GB={row['GB_conc']:.4f} | exp noneq={row['experimental_nonequilibrium_measure']:.4f}")
        plot_single_g2_panel(panel, OUTLIER_OUTPUT_DIR / f'outlier_{rank:02d}_{safe_name}_g2.pdf')
else:
    print('Final outlier export disabled.')
